# 09 · UDFs & pandas UDFs

A **UDF** (user-defined function) lets you run your own Python code on each row when no built-in
function does what you need. They're powerful — and the #1 cause of slow PySpark jobs. Learn both
how to write them **and** when to avoid them.

In [ ]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [ ]:
import time
import pandas as pd

customers = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA}/products.csv", header=True, inferSchema=True)
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)

## A plain Python UDF

In [ ]:
@F.udf(returnType=T.StringType())
def initials(name):
    if name is None:                          # UDFs receive None for nulls — handle it!
        return None
    return "".join(part[0] for part in name.split())

customers.select("name", initials("name").alias("initials")).show(5)

## Why UDFs are slow
```
  JVM executor                         Python worker
  ┌───────────┐  serialise each row   ┌──────────────┐
  │ row data  │ ────────────────────► │ your function│
  │           │ ◄──────────────────── │              │
  └───────────┘  deserialise result   └──────────────┘
```
* Every row is shipped from the JVM to a Python process and back.
* The optimizer **can't see inside** your function, so it can't optimise around it.

Let's measure. (`write.format("noop")` is a handy trick: it runs the full computation but writes nothing.)

In [ ]:
big = (spark.range(5_000_000)
       .withColumn("s", F.concat(F.lit("user_"), (F.col("id") % 1000).cast("string"))))

def timed(df, label):
    df.write.format("noop").mode("overwrite").save()      # warm-up run (starts Python workers etc.)
    t = time.time()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label:<14}{time.time() - t:6.2f}s")

shout_udf = F.udf(lambda s: s.upper() + "!", T.StringType())

@F.pandas_udf(T.StringType())
def shout_pandas(s: pd.Series) -> pd.Series:
    return s.str.upper() + "!"

timed(big.select(F.concat(F.upper("s"), F.lit("!"))), "built-in")
timed(big.select(shout_pandas("s")), "pandas UDF")
timed(big.select(shout_udf("s")), "python UDF")

The built-in version wins comfortably — and on a real cluster with complex functions the gap is
usually far bigger. (Your exact numbers will vary run to run.)

> In older Spark versions plain Python UDFs sent rows one at a time and were often 10× slower than
> pandas UDFs. Recent Spark (4.x) batches them with Apache Arrow by default
> (`spark.sql.execution.pythonUDF.arrow.enabled`), so the two UDF types are now much closer.
> The lesson is the same: **built-in first**.

## pandas UDFs (vectorised UDFs)
A `pandas_udf` receives a whole **batch** of rows as a `pandas.Series` (transferred efficiently with
Apache Arrow), so you write vectorised pandas/numpy code — use it when you really need Python
(e.g. a library like `scipy` or a trained ML model).

In [ ]:
@F.pandas_udf("double")
def zscore_global(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()     # mean/std of the *batch*, not the whole column!

products.select("product_name", "price", F.round(zscore_global("price"), 2).alias("z")).show(5)

## Grouped pandas: `applyInPandas`
Runs a function on each **group** as a full pandas DataFrame — great for per-group logic that's awkward
in Spark (e.g. fitting a small model per customer). You must declare the output schema.

In [ ]:
def add_share(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf["share_of_customer_qty"] = pdf.quantity / pdf.quantity.sum()
    return pdf

(orders.select("customer_id", "order_id", "quantity")
       .groupBy("customer_id")
       .applyInPandas(add_share, schema="customer_id int, order_id int, quantity int, share_of_customer_qty double")
       .show(5))

## The decision ladder
1. **Built-in function** (`F.*`) — always try first. Search the docs!
2. **SQL / higher-order functions** (`F.expr`, `F.transform`, …)
3. **pandas UDF** / `applyInPandas` / `mapInPandas`
4. **Plain Python UDF** — last resort for row-by-row Python logic

### Exercise 1
Write a **Python UDF** `mask_email` that turns `"alice.smith1@gmail.com"` into `"a**********@gmail.com"`: keep the first character, replace every other character **before the `@`** with `*`, keep the `@domain` part. Create `masked` = `customers` plus column **`email_masked`**.

In [ ]:
@F.udf(returnType=T.StringType())
def mask_email(email):
    ...  # TODO

masked = None  # TODO

In [ ]:
# Check — run me after your answer
assert masked.filter(F.col("customer_id") == 1).first().email_masked == 'd***********@outlook.com'
print("Correct!")

### Exercise 2
Now produce the **same** column using **only built-in functions** (no UDF) → `masked_builtin`.

*Hint:* `F.expr` with SQL functions `substr`, `instr`, `repeat`, `concat`.

In [ ]:
masked_builtin = None  # TODO

In [ ]:
# Check — run me after your answer
diff = masked.select("customer_id", "email_masked").exceptAll(masked_builtin.select("customer_id", "email_masked"))
assert diff.count() == 0, "results differ from the UDF version"
print("Correct!")

### Exercise 3
Write a **pandas UDF** `add_vat` that multiplies a price series by 1.2. Create `vat` = `products` plus column **`price_vat`**.

In [ ]:
@F.pandas_udf("double")
def add_vat(s: pd.Series) -> pd.Series:
    ...  # TODO

vat = None  # TODO

In [ ]:
# Check — run me after your answer
assert abs(vat.agg(F.sum("price_vat")).first()[0] - 5711.88) < 0.05
print("Correct!")

---
Stuck? Worked answers (with outputs) are in [`../solutions/09_udfs_and_pandas_udfs.ipynb`](../solutions/09_udfs_and_pandas_udfs.ipynb).